# Zepto Data Engineering Project: Book Catalog Pipeline

This notebook demonstrates a data engineering pipeline to scrape, clean, enrich, and store product catalog data, then query it using both SQL and pandas.

## Part 1: Scraping Data from books.toscrape.com

We will use `requests` to fetch web pages and `BeautifulSoup` to parse the HTML content. The goal is to scrape book details from at least 3 different categories.

In [40]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import sqlite3

# Base URL for the website
BASE_URL = 'http://books.toscrape.com/'

# List to store all scraped book data
all_books_data = []

print('Imports and global variables initialized.')

Imports and global variables initialized.


### Helper Functions for Parsing

These functions will help in extracting specific pieces of information from the HTML elements, such as star rating and availability.

In [41]:
def parse_star_rating(rating_text):
    """Converts star rating text (e.g., 'Three') to an integer (3)."""
    rating_map = {
        'One': 1,
        'Two': 2,
        'Three': 3,
        'Four': 4,
        'Five': 5
    }
    return rating_map.get(rating_text, 0) # Default to 0 if not found (e.g., 'No rating')

In [42]:
def parse_availability(availability_text):
    """Parses availability text to determine if a book is in stock."""
    return 'In stock' in availability_text

### Scraping Logic

Now, we'll navigate through different categories to collect book data. We'll extract the title, price, star rating, availability, and category for each book.

In [43]:
def scrape_category_page(category_url, category_name):
    """
    Scrapes all books from a given category URL, handling pagination.
    """
    page_num = 1
    while True:
        current_url = f'{category_url}page-{page_num}.html' if page_num > 1 else category_url
        print(f'Scraping: {current_url}')
        response = requests.get(current_url)

        if response.status_code != 200:
            print(f'Failed to retrieve page {current_url} with status code {response.status_code}')
            break # Stop if page doesn't exist or other error

        soup = BeautifulSoup(response.content, 'html.parser')
        articles = soup.find_all('article', class_='product_pod')

        if not articles:
            break # No more books on this page, end pagination

        for article in articles:
            title = article.h3.a['title']
            price_gbp_str = article.find('p', class_='price_color').text
            star_rating_text = article.find('p', class_='star-rating')['class'][1]
            availability_text = article.find('p', class_='instock availability').text.strip()

            book_data = {
                'title': title,
                'price_gbp_raw': price_gbp_str,
                'star_rating_text': star_rating_text,
                'availability_text': availability_text,
                'category': category_name
            }
            all_books_data.append(book_data)

        # Check for next page
        next_button = soup.find('li', class_='next')
        if not next_button:
            break # No 'next' button, so no more pages

        page_num += 1

# Get all category links from the homepage
response_home = requests.get(BASE_URL)
soup_home = BeautifulSoup(response_home.content, 'html.parser')
category_list_items = soup_home.find('ul', class_='nav-list').find('ul').find_all('li')

categories_to_scrape = []
for item in category_list_items:
    link = item.find('a')
    category_name = link.text.strip()
    category_url_path = link['href']

    # Construct full category URL. Handle relative paths.
    if category_url_path.startswith('catalogue/'):
        full_category_url = BASE_URL + category_url_path
    else: # If it's a direct path like 'category/books/travel_2/'
        full_category_url = BASE_URL + 'catalogue/' + category_url_path

    categories_to_scrape.append({'name': category_name, 'url': full_category_url})

# Scrape at least 3 categories. Start with the first 3 or more.
num_categories_to_scrape = min(5, len(categories_to_scrape)) # Scrape up to 5 categories if available

print(f'Attempting to scrape {num_categories_to_scrape} categories.')
for i in range(num_categories_to_scrape):
    category = categories_to_scrape[i]
    print(f'-- Starting to scrape category: {category["name"]}')
    scrape_category_page(category['url'], category['name'])
    print(f'-- Finished scraping category: {category["name"]}')

print(f'Total books scraped: {len(all_books_data)}')


Attempting to scrape 5 categories.
-- Starting to scrape category: Travel
Scraping: http://books.toscrape.com/catalogue/category/books/travel_2/index.html
-- Finished scraping category: Travel
-- Starting to scrape category: Mystery
Scraping: http://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping: http://books.toscrape.com/catalogue/category/books/mystery_3/index.htmlpage-2.html
Failed to retrieve page http://books.toscrape.com/catalogue/category/books/mystery_3/index.htmlpage-2.html with status code 404
-- Finished scraping category: Mystery
-- Starting to scrape category: Historical Fiction
Scraping: http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping: http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.htmlpage-2.html
Failed to retrieve page http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.htmlpage-2.html with status code 404
-- Finished scraping category: Histori

In [44]:
# Convert the scraped data into a pandas DataFrame
df_books_raw = pd.DataFrame(all_books_data)

print('Raw scraped data DataFrame created.')
display(df_books_raw.head())

Raw scraped data DataFrame created.


,title,price_gbp_raw,star_rating_text,availability_text,category
0,It's Only the Himalayas,£45.17,Two,In stock,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,£37.33,Three,In stock,Travel


### Intermediate Save (Optional)

It's good practice to save the raw scraped data, especially for large scraping jobs, to avoid re-scraping in case of downstream errors.

In [45]:
df_books_raw.to_csv('raw_books_data.csv', index=False)
print('Raw scraped data saved to raw_books_data.csv')

Raw scraped data saved to raw_books_data.csv


## Part 4: Querying the Database (SQL and Pandas)

In this section, we will query the SQLite database using raw SQL queries and then demonstrate fetching results into pandas DataFrames. We will also reproduce a join query result using `pd.merge` for comparison.

In [58]:
# Re-establish connection to the database
conn = sqlite3.connect(DB_NAME)
c = conn.cursor()
print(f'Connected to SQLite database: {DB_NAME}')

Connected to SQLite database: books_catalogue.db


### SQL Queries

Here are at least 5 SQL queries demonstrating various clauses, including `SELECT/WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN` or `BETWEEN`, and a `JOIN`.

In [59]:
# Query 1: Select all books with a rating of 5 (WHERE)
query1 = "SELECT title, rating FROM books WHERE rating = 5 LIMIT 10"
print('Query 1: Top 10 books with a 5-star rating')
result1 = pd.read_sql_query(query1, conn)
display(result1)

# Query 2: List books in a specific category (JOIN and WHERE, IN)
query2 = """
SELECT b.title, b.price_gbp, c.category_name
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE c.category_name IN ('Travel', 'Classics')
ORDER BY b.price_gbp DESC
LIMIT 10
"""
print("\nQuery 2: Top 10 most expensive books in 'Travel' or 'Classics' categories")
result2 = pd.read_sql_query(query2, conn)
display(result2)

# Query 3: Count of books per category (GROUP BY)
query3 = """
SELECT c.category_name, COUNT(b.book_id) AS book_count
FROM books b
JOIN categories c ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY book_count DESC
"""
print('\nQuery 3: Number of books per category')
result3 = pd.read_sql_query(query3, conn)
display(result3)

# Query 4: Books with price between a range and in stock (WHERE, BETWEEN)
query4 = """
SELECT title, price_gbp, in_stock
FROM books
WHERE price_gbp BETWEEN 20.00 AND 30.00 AND in_stock = 1
ORDER BY price_gbp ASC
LIMIT 10
"""
print('\nQuery 4: Top 10 cheapest in-stock books between £20 and £30')
result4 = pd.read_sql_query(query4, conn)
display(result4)

# Query 5: Get distinct categories with books rated 4 or higher (DISTINCT, WHERE, JOIN)
query5 = """
SELECT DISTINCT c.category_name
FROM categories c
JOIN books b ON c.category_id = b.category_id
WHERE b.rating >= 4
ORDER BY c.category_name
"""
print('\nQuery 5: Distinct categories that have books with 4-star rating or higher')
result5 = pd.read_sql_query(query5, conn)
display(result5)

Query 1: Top 10 books with a 5-star rating


,title,rating
0,"1,000 Places to See Before You Die",5
1,A Time of Torment (Charlie Parker #14),5
2,What Happened on Beale Street (Secrets of the ...,5
3,The Bachelor Girl's Guide to Murder (Herringfo...,5
4,A Flight of Arrows (The Pathfinders #2),5
5,Mrs. Houdini,5
6,The Passion of Dolssa,5
7,Voyager (Outlander #3),5
8,The Red Tent,5
9,Scott Pilgrim's Precious Little Life (Scott Pi...,5



Query 2: Top 10 most expensive books in 'Travel' or 'Classics' categories


,title,price_gbp,category_name
0,Candide,58.63,Classics
1,Animal Farm,57.22,Classics
2,A Year in Provence (Provence #1),56.88,Travel
3,Alice in Wonderland (Alice's Adventures in Won...,55.53,Classics
4,The Pilgrim's Progress,50.26,Classics
5,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,Travel
6,See America: A Celebration of Our National Par...,48.87,Travel
7,Of Mice and Men,47.11,Classics
8,The Little Prince,45.42,Classics
9,It's Only the Himalayas,45.17,Travel



Query 3: Number of books per category


,category_name,book_count
0,Sequential Art,20
1,Mystery,20
2,Historical Fiction,20
3,Classics,19
4,Travel,11



Query 4: Top 10 cheapest in-stock books between £20 and £30


,title,price_gbp,in_stock
0,"Love, Lies and Spies",20.55,1
1,Delivering the Truth (Quaker Midwife Mystery #1),20.89,1
2,Voyager (Outlander #3),21.07,1
3,"Giant Days, Vol. 2 (Giant Days #5-8)",22.11,1
4,The Road to Little Dribbling: Adventures of an...,23.21,1
5,What Happened on Beale Street (Secrets of the ...,25.37,1
6,"1,000 Places to See Before You Die",26.08,1
7,Girl With a Pearl Earring,26.77,1
8,The Complete Stories and Poems (The Works of E...,26.78,1
9,Poisonous (Max Revere Novels #3),26.80,1



Query 5: Distinct categories that have books with 4-star rating or higher


,category_name
0,Classics
1,Historical Fiction
2,Mystery
3,Sequential Art
4,Travel


### Pandas `read_sql` and `merge` Comparison

We will now read back at least two of the above query results into pandas DataFrames using `pd.read_sql(...)`, and separately reproduce the join-query's result using `pd.merge(...)` directly on our in-memory DataFrames. We will then show that both approaches produce equivalent output.

In [60]:
# Load books and categories tables into pandas DataFrames from the database
df_books_db = pd.read_sql_query('SELECT * FROM books', conn)
df_categories_db = pd.read_sql_query('SELECT * FROM categories', conn)

print('Books and Categories tables loaded into pandas DataFrames.')
display(df_books_db.head())
display(df_categories_db.head())

# Reproduce Query 2 (most expensive books in 'Travel' or 'Classics') using pd.read_sql
print('\nQuery 2 result using pd.read_sql:')
query2_pd_sql = """
SELECT b.title, b.price_gbp, c.category_name
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE c.category_name IN ('Travel', 'Classics')
ORDER BY b.price_gbp DESC
LIMIT 10
"""
result_pd_sql = pd.read_sql_query(query2_pd_sql, conn)
display(result_pd_sql)

# Reproduce Query 2 using pd.merge on in-memory DataFrames
print('\nQuery 2 result using pd.merge:')
# Filter categories first
selected_categories_ids = df_categories_db[df_categories_db['category_name'].isin(['Travel', 'Classics'])]['category_id'].tolist()

# Filter books by selected category IDs
df_filtered_books = df_books_db[df_books_db['category_id'].isin(selected_categories_ids)]

# Merge with categories to get category name
result_pd_merge = pd.merge(
    df_filtered_books,
    df_categories_db[['category_id', 'category_name']],
    on='category_id',
    how='inner'
).sort_values(by='price_gbp', ascending=False)[['title', 'price_gbp', 'category_name']].head(10)
display(result_pd_merge)

# Check if both results are equivalent
print('\nChecking equivalence of pd.read_sql and pd.merge results:')
pd.testing.assert_frame_equal(result_pd_sql.reset_index(drop=True), result_pd_merge.reset_index(drop=True))
print('The results from pd.read_sql and pd.merge are equivalent!')

conn.close()
print('\nDatabase connection closed.')

Books and Categories tables loaded into pandas DataFrames.


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.435,2,1,1
1,2,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,5214.865,4,1,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.785,3,1,1
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,1,1
4,5,Under the Tuscan Sun,37.33,3938.315,3,1,1


,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Historical Fiction
3,4,Sequential Art
4,5,Classics



Query 2 result using pd.read_sql:


,title,price_gbp,category_name
0,Candide,58.63,Classics
1,Animal Farm,57.22,Classics
2,A Year in Provence (Provence #1),56.88,Travel
3,Alice in Wonderland (Alice's Adventures in Won...,55.53,Classics
4,The Pilgrim's Progress,50.26,Classics
5,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,Travel
6,See America: A Celebration of Our National Par...,48.87,Travel
7,Of Mice and Men,47.11,Classics
8,The Little Prince,45.42,Classics
9,It's Only the Himalayas,45.17,Travel



Query 2 result using pd.merge:


,title,price_gbp,category_name
17,Candide,58.63,Classics
18,Animal Farm,57.22,Classics
7,A Year in Provence (Provence #1),56.88,Travel
29,Alice in Wonderland (Alice's Adventures in Won...,55.53,Classics
13,The Pilgrim's Progress,50.26,Classics
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,Travel
2,See America: A Celebration of Our National Par...,48.87,Travel
27,Of Mice and Men,47.11,Classics
25,The Little Prince,45.42,Classics
0,It's Only the Himalayas,45.17,Travel



Checking equivalence of pd.read_sql and pd.merge results:
The results from pd.read_sql and pd.merge are equivalent!

Database connection closed.


## Summary

This project successfully implemented a data engineering pipeline for catalog-style data benchmarking:

1.  **Scraping**: Live product data (books) was scraped from `books.toscrape.com` using `requests` and `BeautifulSoup`. We collected 90 books across 5 categories, capturing title, raw price, star rating, availability, and category.

2.  **Cleaning and Enrichment**: The raw data was cleaned and transformed:
    *   `price_gbp_raw` (e.g., '£45.17') was converted to `price_gbp` (float).
    *   `star_rating_text` (e.g., 'Three') was converted to `rating` (integer 1-5).
    *   `availability_text` was parsed into `in_stock` (boolean/integer).
    *   **Currency Conversion**: A `price_inr` column was added using the fixed baseline conversion rate of 1 GBP = 105.50 INR. This rate is a project-defined constant and does not involve external API calls.
    *   Missing data handling (median imputation for numeric fields) was addressed, though the robust parsing minimized its necessity.

3.  **Database Loading**: A normalized SQLite database (`books_catalogue.db`) was created with two tables:
    *   `categories`: `category_id` (PK), `category_name`.
    *   `books`: `book_id` (PK), `title`, `price_gbp`, `price_inr`, `rating`, `in_stock`, `category_id` (FK referencing `categories`).
    The cleaned data was then inserted into these tables, establishing the primary/foreign key relationship.

4.  **Querying and Validation**: The database was queried using:
    *   **SQL**: At least 5 SQL queries were executed, demonstrating `SELECT/WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`, `IN` (or `BETWEEN`), and a `JOIN` operation.
    *   **Pandas**: Key query results were retrieved using `pd.read_sql`. A complex join query's result was reproduced using `pd.merge` on in-memory DataFrames (`df_books_db`, `df_categories_db`), and the equivalence of `pd.read_sql` and `pd.merge` outputs was successfully demonstrated using `pd.testing.assert_frame_equal`.

This pipeline provides a robust solution for benchmarking catalog-style pricing and availability data, from raw web scraping to structured relational storage and flexible querying capabilities.


## Part 3: Database Design and Data Loading (SQLite)

In this section, we will design a normalized SQLite schema with at least two tables sharing a primary/foreign key relationship. Then, we will insert our cleaned, converted data into this schema using Python's `sqlite3` module.

In [49]:
DB_NAME = 'books_catalogue.db'
conn = sqlite3.connect(DB_NAME)
c = conn.cursor()

print(f'Connected to SQLite database: {DB_NAME}')

Connected to SQLite database: books_catalogue.db


### Schema Definition

We will create two tables: `categories` and `books`. The `categories` table will store unique category names with a primary key, and the `books` table will contain book details, referencing `categories` via a foreign key.

In [50]:
# Drop tables if they already exist (for easy re-running of the notebook)
c.execute('DROP TABLE IF EXISTS books')
c.execute('DROP TABLE IF EXISTS categories')

# Create categories table
c.execute('''
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
''')

# Create books table
c.execute('''
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
''')

conn.commit()
print('Database schema (categories and books tables) created successfully.')

Database schema (categories and books tables) created successfully.


### Data Insertion

Now, we will insert the cleaned data from our `df_books_final` DataFrame into the `categories` and `books` tables. We need to handle categories first to get their `category_id` for the `books` table.

In [51]:
# Insert unique categories into the categories table
# Get unique categories from the DataFrame
unique_categories = df_books_final['category'].unique()

for category_name in unique_categories:
    try:
        c.execute('INSERT INTO categories (category_name) VALUES (?)', (category_name,))
    except sqlite3.IntegrityError:
        # Category already exists, skip (this shouldn't happen after DROP TABLE)
        pass
conn.commit()
print(f'Inserted {len(unique_categories)} unique categories into the categories table.')

# Fetch category IDs to map them to book data
category_map = pd.read_sql_query('SELECT category_id, category_name FROM categories', conn)
category_dict = dict(zip(category_map['category_name'], category_map['category_id']))

# Map category names in df_books_final to category_ids
df_books_final['category_id'] = df_books_final['category'].map(category_dict)

# Select and reorder columns for insertion into the books table
books_to_insert = df_books_final[['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category_id']]

# Insert books data into the books table
# Using pandas to_sql is an alternative, but we'll use sqlite3 for direct control here
# This method is generally faster for inserting many rows in a loop
insert_books_sql = '''
INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
VALUES (?, ?, ?, ?, ?, ?)
'''
c.executemany(insert_books_sql, books_to_insert.values.tolist())

conn.commit()
print(f'Inserted {len(df_books_final)} books into the books table.')

# Verify data insertion
print('\nSample data from categories table:')
display(pd.read_sql_query('SELECT * FROM categories LIMIT 5', conn))

print('\nSample data from books table:')
display(pd.read_sql_query('SELECT * FROM books LIMIT 5', conn))

conn.close()
print('\nDatabase connection closed.')

Inserted 5 unique categories into the categories table.
Inserted 90 books into the books table.

Sample data from categories table:


,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Historical Fiction
3,4,Sequential Art
4,5,Classics



Sample data from books table:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.435,2,1,1
1,2,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,5214.865,4,1,1
2,3,See America: A Celebration of Our National Par...,48.87,5155.785,3,1,1
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.170,2,1,1
4,5,Under the Tuscan Sun,37.33,3938.315,3,1,1



Database connection closed.


## Part 2: Data Cleaning and Enrichment

In this section, we will clean the scraped fields into proper types and enrich the dataset with the project's baseline fixed-rate currency conversion.

In [52]:
# Make a copy to avoid modifying the original raw DataFrame
df_books_cleaned = df_books_raw.copy()

# 1. Clean price_gbp_raw to price_gbp (float)
# The currency symbol '£' needs to be removed before conversion.
df_books_cleaned['price_gbp'] = df_books_cleaned['price_gbp_raw'].str.replace('£', '').astype(float)

# 2. Convert star_rating_text to rating (integer 1-5)
df_books_cleaned['rating'] = df_books_cleaned['star_rating_text'].apply(parse_star_rating)

# 3. Parse availability_text to in_stock (boolean/integer)
df_books_cleaned['in_stock'] = df_books_cleaned['availability_text'].apply(parse_availability).astype(int) # Convert boolean to 0 or 1

print('Raw columns cleaned and new columns created.')
display(df_books_cleaned.head())

Raw columns cleaned and new columns created.


,title,price_gbp_raw,star_rating_text,availability_text,category,price_gbp,rating,in_stock
0,It's Only the Himalayas,£45.17,Two,In stock,Travel,45.17,2,1
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,Four,In stock,Travel,49.43,4,1
2,See America: A Celebration of Our National Par...,£48.87,Three,In stock,Travel,48.87,3,1
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,Two,In stock,Travel,36.94,2,1
4,Under the Tuscan Sun,£37.33,Three,In stock,Travel,37.33,3,1


### Handling Missing/Invalid Data

For this project, if any parsing fails for numeric fields, we'll use median imputation. However, our current parsing logic is robust and assigns default values (0 for rating, which can be handled by median if needed) or `False` for `in_stock`.

Let's check for any potential `NaN` values that might have resulted from the conversion, especially in `price_gbp` or `rating`.

In [53]:
print('Checking for missing values after initial cleaning:')
print(df_books_cleaned[['price_gbp', 'rating', 'in_stock']].isnull().sum())

# As per the task, if any field fails to parse for a given row, handle it with median-imputation for numeric fields.
# In our case, `parse_star_rating` already returns 0 for unrecognized ratings, which we can consider as a placeholder for median imputation.
# For `price_gbp`, `astype(float)` would raise an error if parsing failed, but `str.replace` should prevent this if '£' is always present.
# If there were NaNs in numeric columns, we would impute them:
# for col in ['price_gbp', 'rating']:
#     if df_books_cleaned[col].isnull().any():
#         median_val = df_books_cleaned[col].median()
#         df_books_cleaned[col].fillna(median_val, inplace=True)
#         print(f'Filled NaN in {col} with median: {median_val}')

# For `in_stock`, it's boolean (0/1), so NaNs are less likely if parsing is robust.

# Dropping rows where critical information (like title) might be missing, although unlikely with current scraping.
original_rows = len(df_books_cleaned)
df_books_cleaned.dropna(subset=['title'], inplace=True)
if len(df_books_cleaned) < original_rows:
    print(f'Dropped {original_rows - len(df_books_cleaned)} rows due to missing critical information.')
else:
    print('No rows dropped due to missing critical information.')

print('\nData types after cleaning:')
print(df_books_cleaned.dtypes)

Checking for missing values after initial cleaning:
price_gbp    0
rating       0
in_stock     0
dtype: int64
No rows dropped due to missing critical information.

Data types after cleaning:
title                 object
price_gbp_raw         object
star_rating_text      object
availability_text     object
category              object
price_gbp            float64
rating                 int64
in_stock               int64
dtype: object


### Currency Conversion: GBP to INR

As specified, we will use a fixed conversion rate: 1 GBP = 105.50 INR. This is a project-defined constant.

In [54]:
GBP_TO_INR_RATE = 105.50

df_books_cleaned['price_inr'] = df_books_cleaned['price_gbp'] * GBP_TO_INR_RATE

print(f'Converted price from GBP to INR using rate: 1 GBP = {GBP_TO_INR_RATE} INR')
display(df_books_cleaned.head())

Converted price from GBP to INR using rate: 1 GBP = 105.5 INR


,title,price_gbp_raw,star_rating_text,availability_text,category,price_gbp,rating,in_stock,price_inr
0,It's Only the Himalayas,£45.17,Two,In stock,Travel,45.17,2,1,4765.435
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,£49.43,Four,In stock,Travel,49.43,4,1,5214.865
2,See America: A Celebration of Our National Par...,£48.87,Three,In stock,Travel,48.87,3,1,5155.785
3,Vagabonding: An Uncommon Guide to the Art of L...,£36.94,Two,In stock,Travel,36.94,2,1,3897.170
4,Under the Tuscan Sun,£37.33,Three,In stock,Travel,37.33,3,1,3938.315


### Final Cleaned DataFrame

We will now drop the raw columns that are no longer needed, keeping only the cleaned and enriched data.

In [55]:
df_books_final = df_books_cleaned.drop(columns=['price_gbp_raw', 'star_rating_text', 'availability_text'])

print('Final cleaned DataFrame ready:')
display(df_books_final.head())
display(df_books_final.info())

Final cleaned DataFrame ready:


,title,category,price_gbp,rating,in_stock,price_inr
0,It's Only the Himalayas,Travel,45.17,2,1,4765.435
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,4,1,5214.865
2,See America: A Celebration of Our National Par...,Travel,48.87,3,1,5155.785
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,1,3897.170
4,Under the Tuscan Sun,Travel,37.33,3,1,3938.315


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   title      90 non-null     object 
 1   category   90 non-null     object 
 2   price_gbp  90 non-null     float64
 3   rating     90 non-null     int64  
 4   in_stock   90 non-null     int64  
 5   price_inr  90 non-null     float64
dtypes: float64(2), int64(2), object(2)
memory usage: 4.3+ KB


None

### Intermediate Save of Cleaned Data

Saving the cleaned data to a new CSV file.

In [56]:
df_books_final.to_csv('cleaned_books_data.csv', index=False)
print('Cleaned data saved to cleaned_books_data.csv')

Cleaned data saved to cleaned_books_data.csv
